# 05. Spark SQL and Optimization

Spark SQL lets you write SQL against DataFrames and benefit from the Catalyst optimizer. This is a critical skill for production analytics workloads.

## Topics
- create temporary views
- run SQL queries
- understand `explain()`
- use caching carefully
- improve performance with partitioning and broadcast joins


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SparkSQLDemo').master('local[*]').getOrCreate()

data = [(1, 'Alice', 2500), (2, 'Bob', 3200), (3, 'Charlie', 2800)]
df = spark.createDataFrame(data, ['id', 'name', 'salary'])
df.createOrReplaceTempView('employees')

spark.sql('SELECT * FROM employees').show()
spark.sql('SELECT name, salary FROM employees WHERE salary > 2700').show()
spark.sql('SELECT AVG(salary) AS avg_salary FROM employees').show()

spark.stop()


## Explain plan

The explain plan shows how Spark will optimize and execute a query. Use it to understand your job before performance tuning.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('ExplainPlan').master('local[*]').getOrCreate()

df = spark.range(1, 1000).toDF('id')
df = df.withColumn('double_id', F.col('id') * 2)
df.filter('double_id > 100').explain()

spark.stop()


## Caching and persistence

Cache when a dataset will be reused multiple times in the same job. Do not cache everything blindly.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('CachingDemo').master('local[*]').getOrCreate()

df = spark.range(1, 10000).toDF('num')
df.cache()
print(df.count())
print(df.filter('num % 2 = 0').count())

spark.stop()


## Performance basics
- avoid unnecessary wide transformations
- broadcast small dimension tables when appropriate
- use Parquet for analytical workloads
- reduce shuffle when possible
- monitor partitions and skew

## Exercise
Use a temp view and run a SQL aggregation query on a sample sales dataset.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SalesSQL').master('local[*]').getOrCreate()

sales = [(1, 'A', 100), (2, 'A', 250), (3, 'B', 90), (4, 'B', 200)]
df = spark.createDataFrame(sales, ['order_id', 'region', 'amount'])
df.createOrReplaceTempView('sales_data')

spark.sql('SELECT region, SUM(amount) AS total_amount FROM sales_data GROUP BY region ORDER BY total_amount DESC').show()

spark.stop()
